In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os

# Go to project root
os.chdir('C:/Users/HomePC/Documents/kifiya/news-sentiment-analysis')

plt.style.use('ggplot')

# Load data
stock_df = pd.read_csv('data/raw/stock_prices_clean.csv', parse_dates=['Date'])
print(f"Loaded {len(stock_df)} rows")
print(f"Stocks: {stock_df['Symbol'].unique().tolist()}")
print(stock_df.head(3))

Loaded 2510 rows
Stocks: ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'TSLA', 'JPM', 'BAC', 'WMT', 'PFE']
        Date       Close        High         Low        Open    Volume Symbol
0 2025-05-15  210.614441  212.118484  208.711985  210.116417  45029500   AAPL
1 2025-05-16  210.425201  211.730038  208.941099  211.520861  54737900   AAPL
2 2025-05-19  207.955002  208.652233  203.452858  207.088445  46140500   AAPL


In [3]:
def sma(data, period):
    return data.rolling(window=period).mean()

def ema(data, period):
    return data.ewm(span=period, adjust=False).mean()

def rsi(data, period=14):
    delta = data.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def macd(data, fast=12, slow=26, signal=9):
    ema_fast = data.ewm(span=fast, adjust=False).mean()
    ema_slow = data.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    histogram = macd_line - signal_line
    return macd_line, signal_line, histogram

print("✅ Helper functions ready")

✅ Helper functions ready


In [4]:
symbol = 'AAPL'
df = stock_df[stock_df['Symbol'] == symbol].copy()
df = df.sort_values('Date').reset_index(drop=True)

df['SMA_20'] = sma(df['Close'], 20)
df['SMA_50'] = sma(df['Close'], 50)
df['EMA_20'] = ema(df['Close'], 20)
df['RSI'] = rsi(df['Close'], 14)
df['MACD'], df['MACD_Signal'], df['MACD_Hist'] = macd(df['Close'])

print(f"{symbol}: {len(df)} days")
print(df[['Date', 'Close', 'SMA_20', 'RSI']].tail(5))

AAPL: 251 days
          Date       Close      SMA_20        RSI
246 2026-05-08  293.050018  272.960016  72.941202
247 2026-05-11  292.679993  274.645944  72.360729
248 2026-05-12  294.799988  276.456357  73.654347
249 2026-05-13  298.869995  278.090619  75.978793
250 2026-05-14  298.209991  279.843242  74.825876
